In [1]:
import pandas as pd
import numpy as np

In [2]:
df_gdp = pd.read_csv("../data/raw/gdp.csv", skiprows=4)

df_gdp = df_gdp[df_gdp["Country Name"] == "Sri Lanka"]

df_gdp = df_gdp.drop(columns=["Country Code", "Indicator Name", "Indicator Code"])

df_gdp_long = df_gdp.melt(id_vars=["Country Name"], var_name="Year", value_name="GDP")

df_gdp_long = df_gdp_long.rename(columns={"Country Name": "Country"})

df_gdp_long = df_gdp_long.dropna()

df_gdp_long["Year"] = df_gdp_long["Year"].astype(int)

df_gdp_long = df_gdp_long.sort_values("Year").reset_index(drop=True)

df_gdp_long.head()

,Country,Year,GDP
0,Sri Lanka,1960,145.928701
1,Sri Lanka,1961,145.900945
2,Sri Lanka,1962,141.383198
3,Sri Lanka,1963,119.352332
4,Sri Lanka,1964,122.941809


In [3]:
df_imports = pd.read_csv("../data/raw/imports.csv", skiprows=4)

df_imports = df_imports[df_imports["Country Name"] == "Sri Lanka"]

df_imports = df_imports.drop(columns=["Country Code", "Indicator Name", "Indicator Code"])

df_imports_long = df_imports.melt(id_vars=["Country Name"], var_name="Year", value_name="Imports")

df_imports_long = df_imports_long.rename(columns={"Country Name": "Country"})

df_imports_long = df_imports_long.dropna()

df_imports_long["Year"] = df_imports_long["Year"].astype(int)

df_imports_long = df_imports_long.sort_values("Year").reset_index(drop=True)

df_imports_long.head()

,Country,Year,Imports
0,Sri Lanka,2000,4.48
1,Sri Lanka,2001,3.50
2,Sri Lanka,2002,3.33
3,Sri Lanka,2003,4.64
4,Sri Lanka,2004,4.17


In [4]:
df_ewaste = pd.read_csv("../data/raw/ewaste.csv")

df_ewaste = df_ewaste[["Country", "Year", "E_Waste_Generation_Million_Metric_Tons"]]

df_ewaste = df_ewaste.rename(columns={
    "E_Waste_Generation_Million_Metric_Tons": "E_waste_MT"
})

# convert to metric tons
df_ewaste["E_waste_MT"] = df_ewaste["E_waste_MT"] * 1_000_000

# select one country (India) and map to Sri Lanka
df_ewaste = df_ewaste[df_ewaste["Country"] == "India"]
df_ewaste["Country"] = "Sri Lanka"

df_ewaste = df_ewaste.dropna()

df_ewaste = df_ewaste.sort_values("Year").reset_index(drop=True)

df_ewaste.head()

,Country,Year,E_waste_MT
0,Sri Lanka,2015,4100000.0
1,Sri Lanka,2016,4370000.0
2,Sri Lanka,2017,4750000.0
3,Sri Lanka,2018,4860000.0
4,Sri Lanka,2019,4780000.0


In [5]:
df_merge = pd.merge(df_gdp_long, df_imports_long, on=["Country", "Year"])

df_final = pd.merge(df_merge, df_ewaste, on=["Country", "Year"])

df_final = df_final.sort_values("Year").reset_index(drop=True)

df_final.head()

,Country,Year,GDP,Imports,E_waste_MT
0,Sri Lanka,2015,4057.715835,4.22,4100000.0
1,Sri Lanka,2016,4149.191908,4.95,4370000.0
2,Sri Lanka,2017,4398.888281,4.73,4750000.0
3,Sri Lanka,2019,4081.947727,4.64,4780000.0
4,Sri Lanka,2020,3847.601377,5.36,4600000.0


In [6]:
# create full year range
full_years = pd.DataFrame({
    "Year": range(df_final["Year"].min(), df_final["Year"].max() + 1)
})

df_final = pd.merge(full_years, df_final, on="Year", how="left")

# forward fill
df_final = df_final.ffill()

df_final["Country"] = "Sri Lanka"

df_final = df_final.sort_values("Year").reset_index(drop=True)

df_final.head(10)

,Year,Country,GDP,Imports,E_waste_MT
0,2015,Sri Lanka,4057.715835,4.22,4100000.0
1,2016,Sri Lanka,4149.191908,4.95,4370000.0
2,2017,Sri Lanka,4398.888281,4.73,4750000.0
3,2018,Sri Lanka,4398.888281,4.73,4750000.0
4,2019,Sri Lanka,4081.947727,4.64,4780000.0
5,2020,Sri Lanka,3847.601377,5.36,4600000.0
6,2021,Sri Lanka,3996.962400,6.20,4700000.0
7,2022,Sri Lanka,3342.636503,3.01,5040000.0


In [7]:
from sklearn.preprocessing import MinMaxScaler

features = df_final[["GDP", "Imports"]]
target = df_final[["E_waste_MT"]]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(features)
y_scaled = scaler_y.fit_transform(target)

X_scaled_df = pd.DataFrame(X_scaled, columns=["GDP", "Imports"])
y_scaled_df = pd.DataFrame(y_scaled, columns=["E_waste_MT"])

print(X_scaled_df.head())
print(y_scaled_df.head())

        GDP   Imports
0  0.676997  0.379310
1  0.763601  0.608150
2  1.000000  0.539185
3  1.000000  0.539185
4  0.699938  0.510972
   E_waste_MT
0    0.000000
1    0.287234
2    0.691489
3    0.691489
4    0.723404


In [8]:
print(df_final.head())
print(df_final.columns)

   Year    Country          GDP  Imports  E_waste_MT
0  2015  Sri Lanka  4057.715835     4.22   4100000.0
1  2016  Sri Lanka  4149.191908     4.95   4370000.0
2  2017  Sri Lanka  4398.888281     4.73   4750000.0
3  2018  Sri Lanka  4398.888281     4.73   4750000.0
4  2019  Sri Lanka  4081.947727     4.64   4780000.0
Index(['Year', 'Country', 'GDP', 'Imports', 'E_waste_MT'], dtype='str')


In [9]:
print(df_final.isnull().sum())

Year          0
Country       0
GDP           0
Imports       0
E_waste_MT    0
dtype: int64


In [10]:
print(df_final["Year"].tolist())

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [11]:
print(len(df_final))

8


In [12]:
print(X_scaled_df.describe())
print(y_scaled_df.describe())

            GDP   Imports
count  8.000000  8.000000
mean   0.654761  0.539185
std    0.319360  0.286153
min    0.000000  0.000000
25%    0.584127  0.478056
50%    0.688468  0.539185
75%    0.822701  0.640282
max    1.000000  1.000000
       E_waste_MT
count    8.000000
mean     0.570479
std      0.304472
min      0.000000
25%      0.470745
50%      0.664894
75%      0.699468
max      1.000000


In [13]:
print(X_scaled_df.shape)
print(y_scaled_df.shape)

(8, 2)
(8, 1)


In [14]:
# split index (80% approx)
split_index = int(len(df_final) * 0.8)

# split dataset
train = df_final.iloc[:split_index]
test = df_final.iloc[split_index:]

print("Train data:")
print(train)

print("\nTest data:")
print(test)

Train data:
   Year    Country          GDP  Imports  E_waste_MT
0  2015  Sri Lanka  4057.715835     4.22   4100000.0
1  2016  Sri Lanka  4149.191908     4.95   4370000.0
2  2017  Sri Lanka  4398.888281     4.73   4750000.0
3  2018  Sri Lanka  4398.888281     4.73   4750000.0
4  2019  Sri Lanka  4081.947727     4.64   4780000.0
5  2020  Sri Lanka  3847.601377     5.36   4600000.0

Test data:
   Year    Country          GDP  Imports  E_waste_MT
6  2021  Sri Lanka  3996.962400     6.20   4700000.0
7  2022  Sri Lanka  3342.636503     3.01   5040000.0


In [15]:
print(train["Year"].min(), train["Year"].max())
print(test["Year"].min(), test["Year"].max())

2015 2020
2021 2022


In [16]:
from sklearn.preprocessing import MinMaxScaler

# features and target
X_train = train[["GDP", "Imports"]]
X_test = test[["GDP", "Imports"]]

y_train = train[["E_waste_MT"]]
y_test = test[["E_waste_MT"]]

In [17]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

In [18]:
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

In [19]:
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

In [20]:
import pandas as pd

print(pd.DataFrame(X_train_scaled, columns=["GDP","Imports"]))
print(pd.DataFrame(y_train_scaled, columns=["E_waste"]))

        GDP   Imports
0  0.381134  0.000000
1  0.547066  0.640351
2  1.000000  0.447368
3  1.000000  0.447368
4  0.425090  0.368421
5  0.000000  1.000000
    E_waste
0  0.000000
1  0.397059
2  0.955882
3  0.955882
4  1.000000
5  0.735294


In [21]:
import numpy as np

def create_sequences(X, y, window_size):
    X_seq = []
    y_seq = []

    for i in range(len(X) - window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])

    return np.array(X_seq), np.array(y_seq)

In [22]:
window_size = 3

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, window_size)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, window_size)

In [23]:
print(X_train_seq.shape)
print(y_train_seq.shape)

(3, 3, 2)
(3, 1)


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\pro

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\pro

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "D:\Work\04_Y4S1\03_RP\Research\new component 2\pro

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

In [ ]:
pip install tensorflow

In [ ]:
pip install tensorflow==2.16.1

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [ ]:
model = Sequential()

model.add(LSTM(50, activation='relu', input_shape=(3, 2)))
model.add(Dense(1))

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='mse')

In [ ]:
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=30,
    verbose=1
)

In [ ]:
print(X_train_seq.shape)
print(y_train_seq.shape)

In [ ]:
y_pred = model.predict(X_train_seq)

In [ ]:
y_pred_actual = scaler_y.inverse_transform(y_pred)
y_actual = scaler_y.inverse_transform(y_train_seq)

In [ ]:
for i in range(len(y_pred_actual)):
    print(f"Predicted: {y_pred_actual[i][0]:.2f} | Actual: {y_actual[i][0]:.2f}")

In [ ]:
from tensorflow.keras.layers import Bidirectional

In [ ]:
model = Sequential()

model.add(Bidirectional(LSTM(50, activation='relu'), input_shape=(3, 2)))
model.add(Dense(1))

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train_seq, y_train_seq,
    epochs=30,
    verbose=1
)

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
# simulate clients
client1_X = X_train_seq[:1]
client1_y = y_train_seq[:1]

client2_X = X_train_seq[1:2]
client2_y = y_train_seq[1:2]

client3_X = X_train_seq[2:]
client3_y = y_train_seq[2:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def create_model():
    model = Sequential()
    model.add(LSTM(50, activation='relu', input_shape=(3,2)))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# each client trains separately

model1 = create_model()
model1.fit(client1_X, client1_y, epochs=5, verbose=0)

model2 = create_model()
model2.fit(client2_X, client2_y, epochs=5, verbose=0)

model3 = create_model()
model3.fit(client3_X, client3_y, epochs=5, verbose=0)

In [ ]:
import numpy as np

weights1 = model1.get_weights()
weights2 = model2.get_weights()
weights3 = model3.get_weights()

avg_weights = []

for w1, w2, w3 in zip(weights1, weights2, weights3):
    avg_weights.append((w1 + w2 + w3) / 3)

In [ ]:
global_model = create_model()
global_model.set_weights(avg_weights)

In [ ]:
y_pred_fl = global_model.predict(X_train_seq)

y_pred_fl_actual = scaler_y.inverse_transform(y_pred_fl)

print(y_pred_fl_actual)

In [ ]:
results = pd.DataFrame({
    "Year": train["Year"][3:].values,  # adjust for window
    "Actual": y_actual.flatten(),
    "LSTM_Pred": y_pred_actual.flatten(),
    "FL_Pred": y_pred_fl_actual.flatten()
})

results.to_csv("../data/processed/results.csv", index=False)

print(results)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
mae = mean_absolute_error(y_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
mape = np.mean(np.abs((y_actual - y_pred_actual) / y_actual)) * 100

print("LSTM Metrics:")
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

In [ ]:
mae_fl = mean_absolute_error(y_actual, y_pred_fl_actual)
rmse_fl = np.sqrt(mean_squared_error(y_actual, y_pred_fl_actual))
mape_fl = np.mean(np.abs((y_actual - y_pred_fl_actual) / y_actual)) * 100

print("\nFL Metrics:")
print("MAE:", mae_fl)
print("RMSE:", rmse_fl)
print("MAPE:", mape_fl)

In [ ]:
mae = mean_absolute_error(y_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
mape = np.mean(np.abs((y_actual - y_pred_actual) / y_actual)) * 100

print("LSTM Metrics:")
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(y_actual, label="Actual")
plt.plot(y_pred_actual, label="Predicted (LSTM)")
plt.legend()
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
import shap


In [ ]:
!pip install shap

In [ ]:
import shap
print("SHAP OK")

In [ ]:
X_flat = X_train_seq.reshape((X_train_seq.shape[0], -1))

In [ ]:
explainer = shap.KernelExplainer(model.predict, X_flat)

In [ ]:
def model_wrapper(X):
    X_reshaped = X.reshape((X.shape[0], 3, 2))  # back to LSTM shape
    return model.predict(X_reshaped)

In [ ]:
explainer = shap.KernelExplainer(model_wrapper, X_flat)

In [ ]:
shap_values = explainer.shap_values(X_flat)

In [ ]:
shap.summary_plot(shap_values, X_flat)

In [ ]:
shap_values = explainer.shap_values(X_flat)

In [ ]:
shap.summary_plot(shap_values, X_flat)

In [ ]:
model.save("../ml/models/lstm_model.h5")

In [ ]:
import joblib

joblib.dump(scaler_X, "../ml/models/scaler_X.pkl")
joblib.dump(scaler_y, "../ml/models/scaler_y.pkl")